# Hafta 2 - Ek veri: termik santral bulunabilirliği, spot gaz, baraj doluluk

**Amaç:** Eylül → Ekim 2026'da aynı termik yükte fiyatın neden yükseldiğini (eğri kayması +3,0 USD/MWh) üç ek veriyle sınamak.

| Soru | Veri (EPİAŞ Şeffaflık) | Beklenen işaret |
|---|---|---|
| İthal kömür santralleri bakımda / arızada mıydı? | Kaynak bazında emre amade kapasite (EAK) + piyasa mesaj sistemi (arıza-bakım bildirimleri) | İthal kömür EAK'sı Ekim'de düşmüş olmalı |
| Spot gaz fiyatı arttı mı? | Doğal gaz günlük referans fiyatı (GRF) | GRF Ekim'de artmış olmalı |
| Barajların su değeri arttı mı? | Baraj aktif doluluk oranı | Doluluk Ekim'de düşmüş olmalı |

**Çalıştırma:** Bu dosyayı `Hafta_02` klasörüne koyun, *Restart & Run All*. Analiz tablosu `02_islenmis/ana_tablo_h02_saatlik.csv` dosyasını kullanır. Sonuçlar `02_islenmis/ek_*.csv` ve `03_grafikler/hafta02/` altına yazılır.

**Not:** Bu uç noktaları ilk kez kullanıyoruz. Bir veri seti gelmezse not defteri uyarı verip devam eder; çıktıyı bana göndermeniz yeterli.

In [ ]:
# 1) Ayarlar
import os, time, json, getpass
from datetime import date, timedelta
from pathlib import Path
import requests, numpy as np, pandas as pd, matplotlib.pyplot as plt

BAS, BIT = date(2025, 9, 1), date.today() - timedelta(days=1)
ISLN = Path("02_islenmis"); OUT = Path("03_grafikler/hafta02"); OUT.mkdir(parents=True, exist_ok=True)
HAM = Path("01_ham_veri"); HAM.mkdir(exist_ok=True)
LOGIN_URL = "https://giris.epias.com.tr/cas/v1/tickets"
API_ROOT  = "https://seffaflik.epias.com.tr/electricity-service"
GAS_ROOT  = "https://seffaflik.epias.com.tr/natural-gas-service"
iso = lambda d: f"{d.isoformat()}T00:00:00+03:00"
print("Aralık:", BAS, "-", BIT)

In [ ]:
# 3) Giriş: TGT (Ticket Granting Ticket) alma
# Kullanıcı adınız kayıt e-postanızdır. Bilgiler ekrana yazılmaz, kaydedilmez.
class Oturum:
    def __init__(self):
        self.kullanici = os.environ.get("EPIAS_USER") or input("EPİAŞ kullanıcı adı (e-posta): ").strip()
        self._sifre    = os.environ.get("EPIAS_PASS") or getpass.getpass("EPİAŞ şifre: ")
        self.tgt = None
        self.tgt_zamani = 0

    def yenile(self):
        r = requests.post(
            LOGIN_URL,
            data={"username": self.kullanici, "password": self._sifre},
            headers={"Content-Type": "application/x-www-form-urlencoded", "Accept": "text/plain"},
            timeout=30,
        )
        if r.status_code not in (200, 201) or not r.text.strip().startswith("TGT-"):
            raise RuntimeError(f"Giriş başarısız (HTTP {r.status_code}). Kullanıcı adı/şifreyi kontrol edin.")
        self.tgt = r.text.strip()
        self.tgt_zamani = time.time()

    def baslik(self):
        # Bilet birkaç saat geçerlidir; 2 saatte bir yenilemek güvenli tarafta kalır
        if self.tgt is None or time.time() - self.tgt_zamani > 2 * 3600:
            self.yenile()
        return {"TGT": self.tgt, "Content-Type": "application/json", "Accept": "application/json"}

oturum = Oturum()
oturum.yenile()
print("Giriş başarılı.")

In [ ]:
# 3) Genel çağrı fonksiyonları (sayfalı yanıtları da toplar)
def cagri(url, govde=None, yontem="POST", deneme=4):
    for i in range(deneme):
        try:
            if yontem == "GET": r = requests.get(url, headers=oturum.baslik(), timeout=120)
            else: r = requests.post(url, headers=oturum.baslik(), json=govde or {}, timeout=120)
            if r.status_code in (401, 403): oturum.yenile(); continue
            if r.status_code == 429 or r.status_code >= 500: time.sleep(2 ** (i + 1)); continue
            if r.status_code >= 400:
                print(f"  HTTP {r.status_code}: {r.text[:600]}"); return None
            return r.json()
        except requests.exceptions.RequestException as e:
            if i == deneme - 1: print("  Bağlantı hatası:", e); return None
            time.sleep(2 ** (i + 1))

def sayfali(url, govde, boyut=1000, en_cok=200):
    """EPİAŞ sayfalı servisleri: page={number,size}. Tüm sayfaları toplar."""
    tum = []
    for n in range(1, en_cok + 1):
        g = dict(govde); g["page"] = {"number": n, "size": boyut}
        j = cagri(url, g)
        if not j: break
        k = j.get("items") or j.get("body", {}).get("items") or []
        tum += k
        tot = (j.get("page") or {}).get("total")
        if not k or len(k) < boyut or (tot is not None and len(tum) >= tot): break
        time.sleep(0.4)
    return pd.DataFrame(tum)

def aylik_parca(fn):
    """Uzun aralıkları aylık parçalara böler."""
    out, d = [], date(BAS.year, BAS.month, 1)
    while d <= BIT:
        son = min(date(d.year + d.month // 12, d.month % 12 + 1, 1) - timedelta(days=1), BIT)
        df = fn(max(d, BAS), son)
        if df is not None and len(df): out.append(df)
        print(f"  {d:%Y-%m}: {0 if df is None else len(df)} kayıt")
        d = son + timedelta(days=1); time.sleep(0.4)
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()

## 4) Emre amade kapasite (EAK), kaynak bazında
Santralin o saatte üretime hazır kapasitesi. Bakım ya da arıza varsa düşer.

In [ ]:
def eak_ay(b, s):
    j = cagri(API_ROOT + "/v1/generation/data/aic", {"startDate": iso(b), "endDate": iso(s), "region": "TR1"})
    return pd.DataFrame(j.get("items", [])) if j else None
eak = aylik_parca(eak_ay)
print(eak.shape, list(eak.columns)[:20])
if len(eak):
    eak["zaman"] = pd.to_datetime(eak["date"], utc=True).dt.tz_convert("Europe/Istanbul").dt.tz_localize(None)
    eak = eak.drop_duplicates("zaman").set_index("zaman").sort_index()
    eak.to_csv(ISLN / "ek_eak_saatlik.csv", encoding="utf-8-sig"); display(eak.tail(3))

## 5) Piyasa mesaj sistemi: arıza ve bakım bildirimleri
Her bildirimde santral, etkilenen kapasite, başlangıç-bitiş ve neden yer alır.

In [ ]:
bolgeler = cagri(API_ROOT + "/v1/markets/data/umm-region-list", yontem="GET")
print("Bölge listesi:", str(bolgeler)[:400])
def bolge_id(j):
    k = (j or {}).get("items") or (j or {}).get("body", {}).get("items") or (j if isinstance(j, list) else [])
    idal = lambda x: x.get("regionId", x.get("id"))
    for x in k:
        ad = " ".join(str(v) for v in x.values()).upper()
        if "TR1" in ad or "TÜRK" in ad or "TURK" in ad: return idal(x)
    return idal(k[0]) if k else 1
REG = bolge_id(bolgeler); print("Kullanılan regionId:", REG)

def umm_ay(b, s):
    govde = {"startDate": iso(b), "endDate": iso(s), "regionId": REG}
    df = sayfali(API_ROOT + "/v1/markets/data/market-message-system", govde)
    if df.empty:   # sayfa parametresi kabul edilmezse sayfasız dene
        j = cagri(API_ROOT + "/v1/markets/data/market-message-system", govde)
        df = pd.DataFrame((j or {}).get("items", []))
    return df
umm = aylik_parca(umm_ay)
print(umm.shape, list(umm.columns))
if len(umm):
    umm = umm.drop_duplicates(subset=[c for c in ["id"] if c in umm] or None)
    umm.drop(columns=[c for c in ["faultDetails"] if c in umm]).to_csv(ISLN / "ek_umm_bildirimler.csv", index=False, encoding="utf-8-sig")
    display(umm.head(5))

## 6) Doğal gaz günlük referans fiyatı (GRF)

In [ ]:
def grf_ay(b, s):
    j = cagri(GAS_ROOT + "/v1/markets/sgp/data/daily-reference-price", {"startDate": iso(b), "endDate": iso(s)})
    return pd.DataFrame(j.get("items", [])) if j else None
grf = aylik_parca(grf_ay)
print(grf.shape, list(grf.columns))
if len(grf):
    tk = [c for c in grf.columns if "date" in c.lower() or "day" in c.lower()][0]
    grf["gun"] = pd.to_datetime(grf[tk], utc=True).dt.tz_convert("Europe/Istanbul").dt.tz_localize(None).dt.normalize()
    grf.to_csv(ISLN / "ek_grf_gunluk.csv", index=False, encoding="utf-8-sig"); display(grf.tail(3))

## 7) Baraj aktif doluluk
Servis tarih parametresi almayabilir; önce tarih aralığıyla, olmazsa tarihsiz denenir.

In [ ]:
baraj = sayfali(API_ROOT + "/v1/dams/data/active-fullness", {"startDate": iso(BAS), "endDate": iso(BIT)})
if baraj.empty: baraj = sayfali(API_ROOT + "/v1/dams/data/active-fullness", {})
print(baraj.shape, list(baraj.columns))
if len(baraj):
    baraj["gun"] = pd.to_datetime(baraj["date"], utc=True).dt.tz_convert("Europe/Istanbul").dt.tz_localize(None).dt.normalize()
    print("Tarih aralığı:", baraj.gun.min(), "-", baraj.gun.max(), "| baraj sayısı:", baraj.get("dam", pd.Series()).nunique())
    baraj.to_csv(ISLN / "ek_baraj_doluluk.csv", index=False, encoding="utf-8-sig")

## 8) Analiz: Eylül ve Ekim 2026'yı (ve Ekim 2025'i) karşılaştır

In [ ]:
ana = pd.read_csv(ISLN / "ana_tablo_h02_saatlik.csv", parse_dates=["zaman"]).set_index("zaman")
ana["kur"] = (ana.ptf / ana.ptf_usd).where(ana.ptf_usd > 1); ana["kur"] = ana.kur.groupby(ana.index.date).transform("median")
DON = {"Eki 2025": ("2025-10-01", "2025-10-31"), "Eyl 2026": ("2026-09-01", "2026-09-30"), "Eki 2026": ("2026-10-01", "2026-10-31")}
sonuc = {}
for ad, (b, s) in DON.items():
    r = {}
    a = ana.loc[b:s]
    if len(eak):
        e = eak.loc[b:s]
        for k, et in [("ithalKomur", "ithal kömür"), ("dogalgaz", "doğal gaz"), ("linyit", "linyit"), ("barajli", "barajlı")]:
            if k in e: r[f"EAK {et} (GW)"] = e[k].mean() / 1000
        if "ithalKomur" in e:
            r["İthal kömür kullanım oranı (%)"] = (a.uretim_importCoal.mean() / e.ithalKomur.mean()) * 100
        if "dogalgaz" in e:
            r["Doğal gaz kullanım oranı (%)"] = ((a.uretim_naturalGas + a.uretim_lng).mean() / e.dogalgaz.mean()) * 100
    if len(grf):
        g = grf.set_index("gun").loc[b:s]; pk = [c for c in g.columns if pd.api.types.is_numeric_dtype(g[c]) and "price" in c.lower()] or [c for c in g.columns if pd.api.types.is_numeric_dtype(g[c])]
        if pk:
            r["GRF (TL/1000 Sm³)"] = g[pk[0]].mean()
            r["GRF yakıt maliyeti (USD/MWh, 175 Sm³)"] = g[pk[0]].mean() / 1000 * 175 / a.kur.median()
    if len(baraj) and baraj.gun.min() <= pd.Timestamp(s):
        bj = baraj[(baraj.gun >= b) & (baraj.gun <= s)]
        if len(bj): r["Baraj aktif doluluk ort. (%)"] = bj.activeFullnessAmount.mean()
    sonuc[ad] = r
sonuc = pd.DataFrame(sonuc); display(sonuc.round(2))
sonuc.to_csv(ISLN / "ek_test_sonuclari.csv", encoding="utf-8-sig")

In [ ]:
# Arıza-bakım bildirimleri: Eylül ve Ekim 2026'da en büyük kapasite kayıpları
if len(umm):
    u = umm.copy()
    for c in ["caseStartDate", "caseEndDate"]:
        if c in u: u[c] = pd.to_datetime(u[c], utc=True, errors="coerce").dt.tz_convert("Europe/Istanbul").dt.tz_localize(None)
    kap = "capacityAtCaseTime" if "capacityAtCaseTime" in u else None
    def aktif_gun(ay_bas, ay_son):
        gunler = pd.date_range(ay_bas, ay_son, freq="D")
        return pd.Series({g: u[(u.caseStartDate <= g + pd.Timedelta(hours=23)) & (u.caseEndDate >= g)][kap].sum() / 1000 for g in gunler})
    if kap and "caseStartDate" in u:
        g9, g10 = aktif_gun("2026-09-01", "2026-09-30"), aktif_gun("2026-10-01", "2026-10-31")
        print(f"Günlük ortalama bildirilen kapasite kaybı: Eylül {g9.mean():.2f} GW | Ekim {g10.mean():.2f} GW")
        ekim = u[(u.caseStartDate <= "2026-10-31") & (u.caseEndDate >= "2026-10-01")]
        ekim = ekim.assign(gun_sayisi=((ekim.caseEndDate.clip(upper=pd.Timestamp("2026-10-31 23:59")) - ekim.caseStartDate.clip(lower=pd.Timestamp("2026-10-01"))).dt.total_seconds() / 86400).clip(lower=0))
        ekim["GWgun"] = ekim[kap] / 1000 * ekim.gun_sayisi
        kol = [c for c in ["powerPlantName", "uevcbName", "reason", kap, "caseStartDate", "caseEndDate", "GWgun"] if c in ekim]
        top = ekim.sort_values("GWgun", ascending=False)[kol].head(20); display(top)
        top.to_csv(ISLN / "ek_umm_ekim_top20.csv", index=False, encoding="utf-8-sig")
        pd.DataFrame({"Eylül": g9.reset_index(drop=True), "Ekim": g10.reset_index(drop=True)}).to_csv(ISLN / "ek_umm_gunluk_kayip.csv", encoding="utf-8-sig")

In [ ]:
# Grafik: ithal kömür EAK ve üretim (günlük), Eylül-Ekim 2026
if len(eak) and "ithalKomur" in eak:
    e = eak.loc["2026-09-01":"2026-10-31"].ithalKomur.resample("D").mean() / 1000
    u_ = ana.loc["2026-09-01":"2026-10-31"].uretim_importCoal.resample("D").mean() / 1000
    fig, ax = plt.subplots(figsize=(10, 4.8))
    ax.plot(e.index, e.values, color="#0a1f6b", lw=2.6, label="İthal kömür EAK (hazır kapasite)")
    ax.plot(u_.index, u_.values, color="#e8590c", lw=2.6, label="İthal kömür üretimi")
    ax.set_ylabel("GW (günlük ort.)"); ax.legend(frameon=False); fig.autofmt_xdate()
    fig.savefig(OUT / "h02_ithal_komur_eak.png", dpi=200, bbox_inches="tight", facecolor="white"); plt.show()
print("Bitti. 02_islenmis/ek_*.csv dosyalarını ve bu not defterini bana gönderin (ya da klasörden okumamı söyleyin).")